# 05 — Modelo principal: 2.5D ResNet-50 con transfer learning

Entrena el modelo central del proyecto (Objetivo O3): 5-fold CV estratificada **por paciente**, mixed precision, attention pooling, transfer learning desde ImageNet.

**Hardware:** RTX 3060 12 GB. **Tiempo total estimado:** ~2.7 h con cache pre-procesado en `data/cache/`.

Configuración utilizada (alineada con `configs/dl_2_5d.yaml`, ya validada en la corrida de referencia):
- `batch_size = 32`
- `slices_per_patient_train = 8`, `slices_per_patient_eval = 16`
- `epochs = 15`, `early_stopping_patience = 4`
- `lr_backbone = 1e-4`, `lr_head = 1e-3`, AdamW, cosine schedule + 1-epoch warmup
- `pos_weight` calculado por fold (≈ 0.18 dado el imbalance 5.71:1 estudios cirrótico:sano)
- Mixed precision (`torch.cuda.amp`) y gradient clipping a 1.0

Si los checkpoints ya existen en `reports/checkpoints/resnet25d_fold{0..4}.pt`, este notebook los **sobrescribe** con seeds idénticas (resultados reproducibles).

In [1]:
import sys, time
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch

from src.data.inventory import build_inventory
from src.data.splits import make_stratified_group_kfold, class_pos_weight
from src.data.dataset import CirrhosisDataModule
from src.models.cnn_2_5d import build_resnet25d
from src.training.train_dl import train_one_fold, _evaluate
from src.training.utils import set_seed, count_parameters

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')
if torch.cuda.is_available():
    print(f'  {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)')

device: cuda
  NVIDIA GeForce RTX 3060 (12.9 GB)


In [2]:
DATA_ROOT = PROJECT_ROOT / 'data' / 'CirrMRI600plus_raw'
CACHE_DIR = PROJECT_ROOT / 'data' / 'cache'
CKPT_DIR  = PROJECT_ROOT / 'reports' / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

inventory = build_inventory(DATA_ROOT)
records = inventory.filter(task='binary')
print(f'records: {len(records)}  cirr={sum(r.is_cirrhotic for r in records)}  healthy={sum(not r.is_cirrhotic for r in records)}')
print(f'cache_dir existe: {CACHE_DIR.exists()} ({sum(1 for _ in CACHE_DIR.glob("*.npz")) if CACHE_DIR.exists() else 0} archivos)')

records: 738  cirr=628  healthy=110
cache_dir existe: True (738 archivos)


## Loop 5-fold

Cada fold reporta su mejor `val_auc` y guarda checkpoint en `reports/checkpoints/resnet25d_fold{k}.pt` + log per-epoch CSV.

**Importante:** la celda siguiente tarda **~2.7 h en total**. Asegúrate de tener energía conectada y no apagar la máquina. Si quieres pausar entre folds, puedes interrumpir y re-lanzar — los folds completos quedan en disco y los re-corre desde cero (no hay resume parcial, pero los `fold_states` previos quedan en variable de memoria si no reinicias el kernel).

In [3]:
fold_states = []
oof_preds = []  # lista de dicts: patient_id, label, proba, fold

t_global = time.time()
for fold in make_stratified_group_kfold(records, task='binary', n_splits=5, seed=42):
    print(f'\n========== FOLD {fold.fold_id} ==========')
    dm = CirrhosisDataModule(
        inventory, fold, task='binary',
        batch_size=32, num_workers=4,
        slices_per_patient_train=8, slices_per_patient_eval=16,
        cache_dir=CACHE_DIR,
    )
    y_tr = np.array([int(r.is_cirrhotic) for r in dm.train_recs])
    pw = class_pos_weight(y_tr)
    print(f'  train_recs={len(dm.train_recs)}  val_recs={len(dm.val_recs)}  pos_weight={pw:.3f}')

    model = build_resnet25d(num_classes=1, in_channels=3, pretrained=True, dropout=0.3, attention_pool='gated')
    total, train_p = count_parameters(model)
    print(f'  params total={total:,} trainables={train_p:,}')

    state = train_one_fold(
        model=model,
        train_loader=dm.train_loader(),
        val_loader=dm.val_loader(),
        fold_id=fold.fold_id,
        epochs=15,
        lr_backbone=1e-4,
        lr_head=1e-3,
        weight_decay=1e-4,
        pos_weight=pw,
        grad_accum_steps=1,
        grad_clip_norm=1.0,
        patience=4,
        output_dir=CKPT_DIR,
        seed=42,
    )
    fold_states.append(state)
    print(f'  → best val AUC={state.best_val_auc:.4f} en epoch {state.best_epoch} ({state.epochs_run} epochs, {state.wallclock_sec/60:.1f} min)')

    # Recargar mejor checkpoint y obtener OOF predictions agregadas por paciente.
    best = torch.load(state.checkpoint_path, map_location=device)
    model.load_state_dict(best['model'])
    _, val_auc, probs, labels, pids = _evaluate(model, dm.val_loader(), device)
    per_patient = {}; per_label = {}
    for pid, p, y in zip(pids, probs, labels, strict=True):
        per_patient.setdefault(pid, []).append(float(p))
        per_label[pid] = int(y)
    for pid, ps in per_patient.items():
        oof_preds.append({'patient_id': pid, 'label': per_label[pid], 'proba': float(np.mean(ps)), 'fold': fold.fold_id})

print(f'\nTotal 5-fold: {(time.time()-t_global)/3600:.2f} h')


========== FOLD 0 ==========
  train_recs=590  val_recs=148  pos_weight=0.175
  params total=24,034,754 trainables=24,034,754
  → best val AUC=0.9987 en epoch 1 (6 epochs, 26.9 min)

========== FOLD 1 ==========
  train_recs=588  val_recs=150  pos_weight=0.176
  params total=24,034,754 trainables=24,034,754
  → best val AUC=1.0000 en epoch 2 (7 epochs, 46.7 min)

========== FOLD 2 ==========
  train_recs=593  val_recs=145  pos_weight=0.174
  params total=24,034,754 trainables=24,034,754
  → best val AUC=0.9986 en epoch 0 (5 epochs, 34.4 min)

========== FOLD 3 ==========
  train_recs=586  val_recs=152  pos_weight=0.177
  params total=24,034,754 trainables=24,034,754
  → best val AUC=0.9906 en epoch 0 (5 epochs, 23.4 min)

========== FOLD 4 ==========
  train_recs=595  val_recs=143  pos_weight=0.174
  params total=24,034,754 trainables=24,034,754
  → best val AUC=1.0000 en epoch 7 (12 epochs, 53.7 min)

Total 5-fold: 3.23 h


In [4]:
# Guardar OOF predictions + timing breakdown
out_dir = PROJECT_ROOT / 'reports' / 'tables'
out_dir.mkdir(parents=True, exist_ok=True)
oof_df = pd.DataFrame(oof_preds)
oof_df.to_csv(out_dir / 'dl_2_5d_preds.csv', index=False)
print(f'OOF saved: {out_dir / "dl_2_5d_preds.csv"} ({len(oof_df)} filas)')

timing = pd.DataFrame([{
    'fold': s.fold_id,
    'epochs_run': s.epochs_run,
    'best_epoch': s.best_epoch,
    'best_val_auc': s.best_val_auc,
    'wallclock_h': s.wallclock_sec / 3600,
} for s in fold_states])
timing.to_csv(PROJECT_ROOT / 'reports' / 'timing.csv', index=False)
print('\nTiming por fold:')
timing

OOF saved: c:\Users\Sebas\Desktop\Talleres IyV\cirrosis-detection\reports\tables\dl_2_5d_preds.csv (392 filas)

Timing por fold:


,fold,epochs_run,best_epoch,best_val_auc,wallclock_h
0,0,6,1,0.998663,0.448079
1,1,7,2,1.000000,0.778645
2,2,5,0,0.998623,0.573861
3,3,5,0,0.990642,0.390440
4,4,12,7,1.000000,0.895129


In [5]:
# Quick-look agregado sobre OOF concatenadas
from sklearn.metrics import roc_auc_score, average_precision_score
y = oof_df['label'].to_numpy()
p = oof_df['proba'].to_numpy()
print(f'AUC-ROC (OOF, n={len(y)}): {roc_auc_score(y, p):.4f}')
print(f'AUC-PR  (OOF, n={len(y)}): {average_precision_score(y, p):.4f}')

AUC-ROC (OOF, n=392): 0.9859
AUC-PR  (OOF, n=392): 0.9977


## Resultado esperado (resultado de la corrida de referencia)

| Fold | best val_auc | epochs | wall-clock |
|---|---|---|---|
| 0 | 1.0000 | 6 (best epoch 1) | 0.47 h |
| 1 | 1.0000 | 7 (best epoch 2) | 0.56 h |
| 2 | 0.9986 | 5 (best epoch 0) | 0.39 h |
| 3 | 0.9906 | 5 (best epoch 0) | 0.39 h |
| 4 | 1.0000 | 12 (best epoch 7) | 0.88 h |
| **Total** | **0.988 AUC OOF** | | **2.71 h** |

Si tu corrida difiere, verifica que (i) `data/cache/` tenga los 738 archivos `.npz`, (ii) la seed sea 42 en todos los puntos, (iii) `torch.backends.cudnn` no esté en modo `deterministic` (lo está `False` por defecto, que es lo correcto para velocidad).

Continúa al notebook **06_evaluation.ipynb** para producir DeLong, ROC/PR y la tabla principal del paper.